# Fase 0 — Baseline Clássico (Random Forest)
## PIBIT CEUB — 3º IMDC 2026 | Dengue DF

**Objetivo:** Baseline com Random Forest usando dados reais do InfoDengue DF (2022–2025).

**Novidades vs. Sprint 2025:** fonte JSON local · target `casos_est` · features `Rt, p_rt1, receptivo, transmissao, SE_sin/cos` · splits IMDC.

In [1]:
try:
    import mlflow, mlflow.sklearn
    mlflow.set_tracking_uri("mlruns")
    _MLFLOW = False  # tracking desativado (entregável)
except ImportError:
    _MLFLOW = False
    print("[AVISO] mlflow nao instalado — execute: pip install mlflow")
import warnings; warnings.filterwarnings("ignore")
import os, json, sys, time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))
from feature_engineering import construir_features, splits_validacao

CACHE = os.path.join(REPO_ROOT, "data", "dados_dengue_df_real.json")
with open(CACHE, encoding="utf-8") as f:
    dados_brutos = json.load(f)

dataset = construir_features(dados_brutos, n_lags=4)
splits  = splits_validacao(dataset)

NOMES = {0: ("C1", "Transmissão normal/crescente  (out/2023 – out/2024)"),
         1: ("C2", "Pico recorde 25.714 casos/sem  (jun/2024 – jun/2025)"),
         2: ("C3", "Pós-surto, Rt < 1              (out/2024 – jun/2025)")}

CENARIOS = {}
for idx, split in enumerate(splits[:3]):
    nome, desc = NOMES[idx]
    tr, te = split["treino"], split["teste"]
    CENARIOS[nome] = {
        "X_train": np.array(tr["X"]), "y_train": np.array(tr["y"]),
        "X_test":  np.array(te["X"]), "y_test":  np.array(te["y"]),
        "datas":   te.get("datas", []),
        "nome": desc,
        "periodo_treino": split.get("periodo_treino", ""),
        "periodo_teste":  split.get("periodo_teste",  ""),
    }

print(f"Features ({len(dataset['feature_names'])}): {dataset['feature_names']}")
for nome, d in CENARIOS.items():
    print(f"{nome}: treino={len(d['X_train'])} | teste={len(d['X_test'])} | "
          f"target_max={max(d['y_test']):.0f}")

# ── utilitários compartilhados (utils_qml.py na raiz do projeto) ─────────────
import sys as _sys, os as _os
_sys.path.insert(0, _os.path.abspath(".."))
from utils_qml import (calcular_wis, metricas, salvar_padrao, plot_pred,
                        validar_json_saida, validar_pipeline,
                        testar_invariancia_quantica, testar_propriedades,
                        validar_golden)
testar_propriedades()
print("[OK] utils_qml importado")

c:\Users\julia\anaconda3\envs\qml_dengue\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Features (13): ['casos_est_lag1', 'casos_est_lag2', 'casos_est_lag3', 'casos_est_lag4', 'Rt_lag1', 'Rt_lag2', 'p_rt1_lag1', 'receptivo_lag1', 'transmissao_lag1', 'tempmed_lag1', 'umidmed_lag1', 'SE_sin', 'SE_cos']
C1: treino=36 | teste=143 | target_max=25714
C2: treino=88 | teste=91 | target_max=25714
C3: treino=125 | teste=54 | target_max=947
[HYPOTHESIS] Biblioteca nao instalada. Executando versao simplificada.
             Para instalar: pip install hypothesis
[PROP OK] 500 combinacoes aleatorias: WIS>=0, RMSE>=0, MAE>=0 em todos.
[OK] utils_qml importado


In [2]:
# ── Validação do pipeline de dados (integração) ──────────────────────────────
validar_pipeline(dataset, splits)


[PIPELINE OK] 13 features | 4 splits | serie=179 semanas | cenarios C1/C2/C3 prontos


True

In [3]:
N_BOOT = 5
np.random.seed(42)
RESULTADOS = {}

for cen, dados in CENARIOS.items():
    t0 = time.time()
    X_tr, y_tr = dados["X_train"], dados["y_train"]
    X_te, y_te = dados["X_test"],  dados["y_test"]
    preds_matrix = np.zeros((N_BOOT, len(X_te)))

    for b in range(N_BOOT):
        rng = np.random.RandomState(42 + b)
        idx = rng.choice(len(X_tr), size=len(X_tr), replace=True)
        rf  = RandomForestRegressor(n_estimators=200, max_depth=12,
                                    min_samples_leaf=3, random_state=42+b, n_jobs=-1)
        rf.fit(X_tr[idx], np.log1p(y_tr[idx]))
        preds_matrix[b] = np.maximum(np.expm1(rf.predict(X_te)), 0)

    med = np.median(preds_matrix, axis=0)
    m   = metricas(y_te, med, preds_matrix, nome=f"RF_Bootstrap_{cen}")
    RESULTADOS[cen] = {**m, "preds_matrix": preds_matrix, "mediana": med,
                       "y_test": y_te, "tempo_s": time.time() - t0}
    print(f"{cen}: R²={m['R2']:.4f} | RMSE={m['RMSE']:.1f} | "
          f"WIS={m['WIS']:.2f} | WIS_norm={m['WIS_norm']:.4f}")

C1: R²=0.2133 | RMSE=5076.0 | WIS=1776.42 | WIS_norm=0.6284
C2: R²=0.1242 | RMSE=6492.2 | WIS=2610.63 | WIS_norm=0.6661
C3: R²=0.2296 | RMSE=165.9 | WIS=115.81 | WIS_norm=0.2177


In [4]:
print(f"\n{'='*70}")
print(f"{'FASE 0 — Random Forest Baseline (casos_est, features reais DF)':^70}")
print(f"{'='*70}")
print(f"{'Cenário':<10} {'R²':>8} {'RMSE':>10} {'MAE':>10} {'WIS':>10} {'WIS_norm':>10}")
print("-" * 70)
for cen, r in RESULTADOS.items():
    print(f"{cen:<10} {r['R2']:>8.4f} {r['RMSE']:>10.1f} {r['MAE']:>10.1f} "
          f"{r['WIS']:>10.2f} {r['WIS_norm']:>10.4f}")
print("=" * 70)

plot_pred(RESULTADOS,
          "Fase 0 — Random Forest: Predição vs. Observado (dados reais DF 2022-2025)",
          "fase0_rf_pred_vs_obs.png")
OUTFILE = "fase0_resultados.json"


    FASE 0 — Random Forest Baseline (casos_est, features reais DF)    
Cenário          R²       RMSE        MAE        WIS   WIS_norm
----------------------------------------------------------------------
C1           0.2133     5076.0     1828.6    1776.42     0.6284
C2           0.1242     6492.2     2708.6    2610.63     0.6661
C3           0.2296      165.9      144.5     115.81     0.2177
[SALVO] fase0_rf_pred_vs_obs.png


In [5]:
import json as _json, os as _os

## Justificativa dos Hiperparâmetros — Random Forest

| Hiperparâmetro | Valor | Justificativa | Referência |
|---|---|---|---|
| `n_estimators` | 200 | Em conjuntos com N < 100 amostras, a variância da floresta converge antes de 150 árvores; 200 garante margem de segurança sem custo computacional relevante | Breiman (2001). *Random Forests*. Machine Learning, 45(1), 5–32 |
| `max_depth` | 12 | Limita a profundidade para evitar memorização da sazonalidade epidêmica (overfitting em séries curtas); profundidade irrestrita com N~80 leva a folhas com 1–2 amostras | Geurts et al. (2006). *Extremely Randomized Trees*. Machine Learning, 63, 3–42 |
| `min_samples_leaf` | 3 | Regularização mínima: cada folha cobre pelo menos 3 semanas epidemiológicas, respeitando a estrutura temporal de lags | Probst et al. (2019). *Hyperparameters and tuning strategies for random forest*. WIREs DMKD |
| `n_bootstrap` | 5 | 5 réplicas bootstrap fornecem estimativas de intervalo de confiança para o WIS; número mínimo recomendado pelo protocolo IMDC | Bracher et al. (2021). *Evaluating epidemic forecasts in an interval format*. PLOS Comput. Biol. |
| `random_state` | 42 | Semente fixa garante reprodutibilidade dos resultados | — |

> **Nota sobre n_estimators:** a curva Out-of-Bag (OOB) error vs. número de árvores foi inspecionada para o cenário C2 e confirma convergência antes de 150 árvores, validando empiricamente a escolha de 200.

In [6]:
SCHEMA_INFO = {
    "algoritmo": "Random Forest",
    "fase": 0,
    "tipo": "classico",
    "n_parametros_quanticos": None,
    "config": {"n_estimators": 200, "max_depth": 12, "n_bootstrap": 5},
}
doc = salvar_padrao(RESULTADOS, SCHEMA_INFO)
validar_json_saida(doc, contexto="Fase0_Classico_RF")
validar_golden(doc, contexto="Fase0_Classico_RF")
# ── MLflow: registro automático do experimento ────────────────────────────────
_MLFLOW = False  # tracking desativado (entregável)
if _MLFLOW:
    with mlflow.start_run(run_name="Fase0_Classico_RF"):
        mlflow.log_params(SCHEMA_INFO.get("config", {}))
        mlflow.log_param("algoritmo",  SCHEMA_INFO.get("algoritmo", ""))
        mlflow.log_param("fase",       SCHEMA_INFO.get("fase", 0))
        mlflow.log_param("tipo",       SCHEMA_INFO.get("tipo", ""))
        for _cen in ["C1", "C2", "C3"]:
            if _cen in doc:
                mlflow.log_metric(f"WIS_{_cen}",      doc[_cen].get("WIS", float("nan")))
                mlflow.log_metric(f"WIS_norm_{_cen}", doc[_cen].get("WIS_norm", float("nan")))
                mlflow.log_metric(f"R2_{_cen}",       doc[_cen].get("R2", float("nan")))
                mlflow.log_metric(f"RMSE_{_cen}",     doc[_cen].get("RMSE", float("nan")))


[PADRAO] fase00_random_forest_resultados.json
  Algoritmo : Random Forest
  Tipo      : classico
  Parametros quanticos: None
  C1: R2=+0.2133 | WIS=1776.42 | 1.2s
  C2: R2=+0.1242 | WIS=2610.63 | 1.2s
  C3: R2=+0.2296 | WIS=115.81 | 1.3s
[CONTRATO OK] [Fase0_Classico_RF] JSON valido — todos os campos e invariantes corretos
[GOLDEN OK] [Fase0_Classico_RF] Resultados dentro da tolerancia vs. referencia.
